<a href="https://colab.research.google.com/github/lgelara/prismaFRL/blob/main/Lucia_prisma_deduplicacion_parte1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📚 PRISMA — Paso 1: Eliminación de Duplicados

Este cuaderno realiza el **primer filtrado de una revisión bibliométrica** siguiendo la metodología PRISMA.  
Elimina artículos duplicados comparando el **título** de cada registro.

### Formatos compatibles
| Formato | Extensión | Fuente típica |
|---------|-----------|---------------|
| Hoja de cálculo | `.csv` | Scopus, WoS, cualquier base |
| BibTeX | `.bib` / `.bibtex` | Zotero, Mendeley, Google Scholar |
| PubMed | `.nbib` | PubMed / MEDLINE |

---
### Instrucciones rápidas
1. Ejecuta la **Celda 1** para instalar dependencias (solo la primera vez).
2. En la **Celda 2**, activa o desactiva los formatos que vas a usar.
3. Ajusta el umbral de similitud si lo necesitas.
4. Ejecuta las celdas restantes en orden.

> **Nota:** Si compartes este cuaderno, cada colaborador solo necesita activar los formatos que tiene.

In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELDA 1 · Instalación de dependencias
#  Ejecuta esta celda UNA SOLA VEZ por sesión de Colab
# ─────────────────────────────────────────────────────────────
!pip install bibtexparser rapidfuzz pandas openpyxl -q
print('✓ Dependencias instaladas correctamente')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 42.4 MB/s eta 0:00:00
✓ Dependencias instaladas correctamente


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELDA 2 · CONFIGURACIÓN  ← EDITA AQUÍ
# ─────────────────────────────────────────────────────────────

# ── Activa (True) o desactiva (False) cada formato según los
#    archivos que vayas a subir:
USAR_CSV   = True     # archivos .csv  (Scopus, WoS, etc.)
USAR_BIB   = True     # archivos .bib / .bibtex  (Zotero, Mendeley)
USAR_NBIB  = True     # archivos .nbib  (PubMed / MEDLINE)

# ── Estrategia de deduplicación:
#    'fuzzy'  → detecta duplicados con pequeñas variaciones tipográficas  ← recomendado
#    'exacto' → solo elimina títulos 100% idénticos (más conservador)
ESTRATEGIA = 'fuzzy'

# ── Umbral de similitud para modo 'fuzzy' (0–100):
#    100 = idénticos | 90 = muy estricto (recomendado) | 85 = tolerante
UMBRAL_SIMILITUD = 90

# ── Columna de título en archivos CSV
#    Déjala vacía ('') para detección automática, o escribe el nombre exacto:
COLUMNA_TITULO_CSV = ''

# ── Mostrar tabla de duplicados al final?
MOSTRAR_DUPLICADOS = True

# ─────────────────────────────────────────────────────────────
formatos_activos = [f for f, v in [('CSV', USAR_CSV), ('BIB/BibTeX', USAR_BIB), ('NBIB/PubMed', USAR_NBIB)] if v]
print(f'Formatos activados : {", ".join(formatos_activos)}')
print(f'Estrategia         : {ESTRATEGIA}  |  Umbral: {UMBRAL_SIMILITUD}%')

Formatos activados : CSV, BIB/BibTeX, NBIB/PubMed
Estrategia         : fuzzy  |  Umbral: 90%


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELDA 3 · Funciones de lectura y deduplicación
#  (no necesitas modificar nada aquí)
# ─────────────────────────────────────────────────────────────
import os, re
import pandas as pd
import bibtexparser
from rapidfuzz import fuzz
from IPython.display import display

# ── Normalización de títulos ──────────────────────────────────
def normalizar(titulo):
    if not isinstance(titulo, str):
        return ''
    t = titulo.lower().strip()
    t = re.sub(r'[^\w\s]', '', t)
    t = re.sub(r'\s+', ' ', t)
    return t

# ── Lectores por formato ──────────────────────────────────────
def leer_csv(ruta):
    df = pd.read_csv(ruta, encoding='utf-8', on_bad_lines='skip')
    if COLUMNA_TITULO_CSV and COLUMNA_TITULO_CSV in df.columns:
        col = COLUMNA_TITULO_CSV
    else:
        candidatos = [c for c in df.columns if 'title' in c.lower() or 'titulo' in c.lower()]
        if not candidatos:
            raise ValueError(
                f'No se encontró columna de título en {os.path.basename(ruta)}.\n'
                f'Columnas disponibles: {list(df.columns)}\n'
                f'→ Especifica el nombre en COLUMNA_TITULO_CSV (Celda 2).'
            )
        col = candidatos[0]
    out = pd.DataFrame()
    out['titulo_original']   = df[col]
    out['autor']             = df.get('Author', df.get('Authors', df.get('author', '')))
    out['año']               = df.get('Year', df.get('year', df.get('Publication Year', '')))
    out['revista']           = df.get('Journal', df.get('Source', df.get('journal', '')))
    out['doi']               = df.get('DOI', df.get('doi', ''))
    out['abstract']          = df.get('Abstract', df.get('abstract', ''))
    out['titulo_normalizado']= out['titulo_original'].apply(normalizar)
    out['fuente_archivo']    = os.path.basename(ruta)
    out['formato']           = 'CSV'
    return out

def leer_bib(ruta):
    with open(ruta, 'r', encoding='utf-8', errors='ignore') as f:
        bd = bibtexparser.load(f)
    registros = []
    for e in bd.entries:
        registros.append({
            'titulo_original' : e.get('title', '').replace('{','').replace('}',''),
            'autor'           : e.get('author', ''),
            'año'             : e.get('year', ''),
            'revista'         : e.get('journal', e.get('booktitle', '')),
            'doi'             : e.get('doi', ''),
            'abstract'        : e.get('abstract', ''),
            'tipo'            : e.get('ENTRYTYPE', ''),
            'clave_bibtex'    : e.get('ID', ''),
        })
    df = pd.DataFrame(registros)
    df['titulo_normalizado'] = df['titulo_original'].apply(normalizar)
    df['fuente_archivo']     = os.path.basename(ruta)
    df['formato']            = 'BIB'
    return df

def leer_nbib(ruta):
    MAPA = {'TI':'titulo_original','AU':'autor','DP':'año',
            'JT':'revista','LID':'doi','AB':'abstract','PT':'tipo','PMID':'pmid'}
    registros, actual, campo_actual = [], {}, None
    with open(ruta, 'r', encoding='utf-8', errors='ignore') as f:
        lineas = f.readlines()
    for linea in lineas:
        linea = linea.rstrip('\n')
        if linea.strip() == '':
            if actual:
                registros.append(actual.copy())
                actual, campo_actual = {}, None
            continue
        m = re.match(r'^([A-Z]{2,4})\s*-\s*(.*)', linea)
        if m:
            campo_actual = m.group(1)
            nombre = MAPA.get(campo_actual, campo_actual.lower())
            val = m.group(2).strip()
            actual[nombre] = actual.get(nombre, '') + (' ' if nombre in actual else '') + val
        elif campo_actual and linea.startswith('      '):
            nombre = MAPA.get(campo_actual, campo_actual.lower())
            actual[nombre] = actual.get(nombre, '') + ' ' + linea.strip()
    if actual:
        registros.append(actual)
    df = pd.DataFrame(registros)
    if 'titulo_original' not in df.columns:
        df['titulo_original'] = ''
    df['titulo_normalizado'] = df['titulo_original'].apply(normalizar)
    df['fuente_archivo']     = os.path.basename(ruta)
    df['formato']            = 'NBIB'
    return df

LECTORES = {'.csv': leer_csv, '.bib': leer_bib,
            '.bibtex': leer_bib, '.nbib': leer_nbib}
ACTIVOS  = {'.csv': USAR_CSV, '.bib': USAR_BIB,
            '.bibtex': USAR_BIB, '.nbib': USAR_NBIB}

# ── Deduplicación ─────────────────────────────────────────────
def deduplicar(df):
    df = df.copy().reset_index(drop=True)
    es_dup   = [False] * len(df)
    dup_de   = [''] * len(df)
    titulos  = df['titulo_normalizado'].tolist()
    marcados = set()
    for i in range(len(titulos)):
        if i in marcados or not titulos[i]:
            continue
        for j in range(i + 1, len(titulos)):
            if j in marcados or not titulos[j]:
                continue
            if ESTRATEGIA == 'exacto':
                igual = titulos[i] == titulos[j]
            else:
                igual = fuzz.token_sort_ratio(titulos[i], titulos[j]) >= UMBRAL_SIMILITUD
            if igual:
                marcados.add(j)
                es_dup[j] = True
                dup_de[j] = df.at[i, 'titulo_original']
    df['_es_duplicado'] = es_dup
    df['_duplicado_de'] = dup_de
    unicos = df[~df['_es_duplicado']].drop(columns=['_es_duplicado','_duplicado_de'])
    dups   = df[df['_es_duplicado']].drop(columns=['_es_duplicado'])
    return unicos.reset_index(drop=True), dups.reset_index(drop=True)

print('✓ Funciones cargadas correctamente')

✓ Funciones cargadas correctamente


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELDA 4 · Subir archivos
# ─────────────────────────────────────────────────────────────
from google.colab import files

tipos_esperados = []
if USAR_CSV:  tipos_esperados.append('.csv')
if USAR_BIB:  tipos_esperados.extend(['.bib', '.bibtex'])
if USAR_NBIB: tipos_esperados.append('.nbib')

print(f'Formatos activos: {", ".join(tipos_esperados)}')
print('Selecciona tus archivos (puedes subir varios a la vez):\n')

archivos_subidos = files.upload()

# Guardar en disco local de Colab
for nombre, contenido in archivos_subidos.items():
    with open(nombre, 'wb') as f:
        f.write(contenido)

print(f'\n✓ {len(archivos_subidos)} archivo(s) subido(s):')
for nombre in archivos_subidos:
    ext = os.path.splitext(nombre)[1].lower()
    estado = '✓ activo' if ext in tipos_esperados else f'⚠ omitido (formato {ext} desactivado)'
    print(f'   {nombre}  →  {estado}')

Formatos activos: .csv, .bib, .bibtex, .nbib
Selecciona tus archivos (puedes subir varios a la vez):



TypeError: 'NoneType' object is not subscriptable

In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELDA 5 · Leer, combinar y deduplicar
# ─────────────────────────────────────────────────────────────
dataframes = []
errores    = []

for nombre in archivos_subidos:
    ext = os.path.splitext(nombre)[1].lower()
    if not ACTIVOS.get(ext, False):
        print(f'  ⏭  {nombre}  (formato desactivado, se omite)')
        continue
    lector = LECTORES.get(ext)
    if lector is None:
        print(f'  ✗  {nombre}  (extensión no reconocida)')
        continue
    try:
        df = lector(nombre)
        dataframes.append(df)
        print(f'  ✓  {nombre}  →  {len(df)} registros')
    except Exception as e:
        errores.append(nombre)
        print(f'  ✗  {nombre}  →  Error: {e}')

if not dataframes:
    raise RuntimeError('No se pudo leer ningún archivo. Revisa los errores de arriba.')

df_total = pd.concat(dataframes, ignore_index=True)
print(f'\nTotal registros combinados: {len(df_total)}')
print(f'Iniciando deduplicación ({ESTRATEGIA}, umbral {UMBRAL_SIMILITUD}%)...')

df_unicos, df_duplicados = deduplicar(df_total)

n_total = len(df_total)
n_u     = len(df_unicos)
n_d     = len(df_duplicados)

print('\n' + '─'*50)
print(f'  Registros identificados  : {n_total}')
print(f'  Duplicados eliminados    : {n_d}  ({n_d/n_total*100:.1f}%)')
print(f'  Registros únicos         : {n_u}')
print('─'*50)
print('\n✓ Deduplicación completada')

  ✓  ieee (2).csv  →  207 registros
  ✓  scopus_lucia (1).csv  →  97 registros
  ✓  wos (2).csv  →  310 registros

Total registros combinados: 614
Iniciando deduplicación (fuzzy, umbral 90%)...

──────────────────────────────────────────────────
  Registros identificados  : 614
  Duplicados eliminados    : 139  (22.6%)
  Registros únicos         : 475
──────────────────────────────────────────────────

✓ Deduplicación completada


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELDA 6 · Vista previa de resultados
# ─────────────────────────────────────────────────────────────

# ── Resumen PRISMA ────────────────────────────────────────────
resumen = pd.DataFrame({
    'Etapa PRISMA'  : ['Identificación (total)', 'Duplicados eliminados', 'Para screening'],
    'n'             : [n_total, n_d, n_u],
    'Nota'          : [
        f'{len(dataframes)} base(s) de datos',
        f'Estrategia: {ESTRATEGIA}, umbral {UMBRAL_SIMILITUD}%',
        'Proceder a evaluación de título y abstract'
    ]
})
print('\n── Resumen PRISMA ───────────────────────────────────')
display(resumen)

# ── Registros por archivo fuente ──────────────────────────────
print('\n── Registros por archivo ────────────────────────────')
display(df_total.groupby('fuente_archivo').size().reset_index(name='registros'))

# ── Primeras filas de registros únicos ───────────────────────
cols_preview = ['titulo_original', 'autor', 'año', 'revista', 'fuente_archivo']
cols_ok = [c for c in cols_preview if c in df_unicos.columns]
print(f'\n── Muestra: primeros 5 registros únicos ─────────────')
display(df_unicos[cols_ok].head())

# ── Duplicados encontrados ────────────────────────────────────
if MOSTRAR_DUPLICADOS and n_d > 0:
    cols_dup = ['titulo_original', '_duplicado_de', 'fuente_archivo']
    cols_dup_ok = [c for c in cols_dup if c in df_duplicados.columns]
    print(f'\n── Duplicados eliminados (primeros 10) ──────────────')
    display(df_duplicados[cols_dup_ok].head(10))
elif n_d == 0:
    print('\nℹ  No se encontraron duplicados.')


── Resumen PRISMA ───────────────────────────────────


,Etapa PRISMA,n,Nota
0,Identificación (total),614,3 base(s) de datos
1,Duplicados eliminados,139,"Estrategia: fuzzy, umbral 90%"
2,Para screening,475,Proceder a evaluación de título y abstract



── Registros por archivo ────────────────────────────


,fuente_archivo,registros
0,ieee (2).csv,207
1,scopus_lucia (1).csv,97
2,wos (2).csv,310



── Muestra: primeros 5 registros únicos ─────────────


,titulo_original,autor,año,revista,fuente_archivo
0,A Multiagent Federated Reinforcement Learning ...,Y. Chu; Z. Wei; X. Fang; S. Chen; Y. Zhou,2022,,ieee (2).csv
1,A Privacy-Preserving Federated Reinforcement L...,T. Yang; X. Feng; S. Cai; Y. Niu; H. Pen,2025,,ieee (2).csv
2,An Instructional Optimization Method Based on ...,R. Zhang; X. Wu; X. Zhang; L. Xu,2025,,ieee (2).csv
3,Collaborative Decision-Making Method for Multi...,S. Li; Y. Jia; F. Yang; Q. Qin; H. Gao; Y. Zhou,2022,,ieee (2).csv
4,Safe and Interpretable Human-Like Planning Wit...,J. Nan; R. Zhang; G. Yin; W. Zhuang; Y. Zhang;...,2025,,ieee (2).csv



── Duplicados eliminados (primeros 10) ──────────────


,titulo_original,_duplicado_de,fuente_archivo
0,Adapting Deep Learning for Content Caching Fra...,Adapting Deep Learning for Content Caching Fra...,scopus_lucia (1).csv
1,Safe and Energy-Efficient Trajectory Planning ...,Safe and Energy-Efficient Trajectory Planning ...,scopus_lucia (1).csv
2,Adaptive Federated Reinforcement Learning With...,Adaptive Federated Reinforcement Learning With...,wos (2).csv
3,A Blockchain-Enabled Cold Start Aggregation Sc...,A Blockchain-Enabled Cold Start Aggregation Sc...,wos (2).csv
4,Personalized Federated Reinforcement Learning ...,Personalized Federated Reinforcement Learning ...,wos (2).csv
5,Federated Reinforcement Learning-Empowered Tas...,Federated Reinforcement Learning-Empowered Tas...,wos (2).csv
6,A Preference-Based Multi-Agent Federated Reinf...,A Preference-Based Multi-Agent Federated Reinf...,wos (2).csv
7,Scenario-Driven Federated Reinforcement Learni...,Scenario-Driven Federated Reinforcement Learni...,wos (2).csv
8,Location Privacy Preservation Crowdsensing Wit...,Location Privacy Preservation Crowdsensing Wit...,wos (2).csv
9,FOVA: Offline Federated Reinforcement Learning...,FOVA: Offline Federated Reinforcement Learning...,wos (2).csv


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELDA 7 · Exportar y descargar resultados
# ─────────────────────────────────────────────────────────────

ARCHIVO_UNICOS     = 'prisma_unicos.csv'
ARCHIVO_DUPLICADOS = 'prisma_duplicados.csv'
ARCHIVO_LOG        = 'prisma_log_deduplicacion.csv'

df_unicos.to_csv(ARCHIVO_UNICOS, index=False, encoding='utf-8-sig')
df_duplicados.to_csv(ARCHIVO_DUPLICADOS, index=False, encoding='utf-8-sig')
resumen.to_csv(ARCHIVO_LOG, index=False, encoding='utf-8-sig')

print('Archivos generados:')
print(f'  ✓  {ARCHIVO_UNICOS}      → {n_u} registros únicos')
print(f'  ✓  {ARCHIVO_DUPLICADOS}  → {n_d} duplicados')
print(f'  ✓  {ARCHIVO_LOG}         → conteos PRISMA')

print('\nDescargando...')
files.download(ARCHIVO_UNICOS)
files.download(ARCHIVO_DUPLICADOS)
files.download(ARCHIVO_LOG)
print('\n✓ Listo. Próximo paso: screening de título y abstract.')

Archivos generados:
  ✓  prisma_unicos.csv      → 475 registros únicos
  ✓  prisma_duplicados.csv  → 139 duplicados
  ✓  prisma_log_deduplicacion.csv         → conteos PRISMA

Descargando...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ Listo. Próximo paso: screening de título y abstract.
